**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/du-2026"], check=True)
    os.chdir("/content/encore/entries/du-2026")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Du, Lai, Hu & Ye (2026) on LINK: development sessions

Paper: Du Z, Lai ZY, Hu L, Ye T. "A Context-Conditional Audit of Trial-Pairing-Dependent Neural
Gain in Motor-Cortex Decoding". bioRxiv preprint posted 2026-08-10,
doi:10.64898/2026.08.04.742876. The claims file (`claims/du-2026.yaml`) was written from that
preprint's full text (main text, legends, Tables 1 to 3, Methods); the supplement was not available.

Dataset: DANDI:001201 (LINK, Temmar et al.), pinned at version **0.251023.2336**, streamed from the
archive. This is the version the paper cites (Methods 4.2, Data availability; protocol, author step
1.3). Two later versions exist (0.261015.2153 and 0.261026.1912); they list the same 312 asset paths
with the same sizes (next cell), so the analysis would read the same files; the pinned version is the cited one.

This notebook tests the claims on the 18 development sessions (nine dates with a CO and an RD
session). Other notebooks of this entry: `01_synthetic.ipynb` (synthetic calibration),
`03_link_estimators.ipynb` (estimator families), `04_link_temporal_capacity.ipynb` (temporal offset
and context capacity), `05_link_confirmation.ipynb` (20 confirmation sessions), `06_mc_rtt.ipynb`
(DANDI:000129), `07_mc_maze.ipynb` (DANDI:000139, data elsewhere). The estimators are ports of the paper's released code (tag v1.0-jne-submission,
commit b042fbe), in `du2026.py`, each function naming its source.

| Claim | Where | Status |
|---|---|---|
| synthetic-template-null, synthetic-signal-ordering, synthetic-pair-breaking | 01 | tested (data generated by the paper's code) |
| link-development-sample, link-neural-only-r2, link-context-ladder-gain, link-context-only-r2, link-phase-only-replacement, link-task-contrast-by-context | this notebook | tested |
| nested-geometry-gain, nested-history-gain, nested-replacement-negative, replacement-source-diagnostics, permutation-negative, replacement-penalty-distance, replacement-ranks-lower, practical-mse-reduction, representative-trial, split-seed-stability, rff-link | this notebook | tested |
| estimator-family-link | 03 | tested |
| temporal-offset, context-capacity-contrast | 04 | tested |
| confirm-geometry-gain, confirm-history-contraction, confirm-replacement, confirm-table1-values | 05 | tested |
| rtt-data, rtt-gain, rff-rtt, estimator-family-nlb | 06 | tested (MC_Maze part of estimator-family-nlb in 07) |
| maze-gain, rff-maze | 07 | tested on MC_Maze (DANDI:000139 version 0.220113.0408, data elsewhere) |

In [ ]:
import collections, resource, time, zlib
import numpy as np
import pandas as pd
import du2026 as d

PAPER = "du-2026"
def claim_seed(claim_id):
    return zlib.crc32(f"{PAPER}/{claim_id}".encode())

T0 = time.time()
assets = d.list_assets(d.LINK_ID, d.LINK_VERSION)
by_date = collections.defaultdict(list)
for a in assets:
    by_date[d.session_date(a["path"])].append(a)
paired_dates = sorted(k for k, v in by_date.items() if len(v) > 1)
dev_assets = [a for k in paired_dates for a in sorted(by_date[k], key=lambda a: a["path"])]
print(f"{len(assets)} assets in DANDI:{d.LINK_ID} version {d.LINK_VERSION}, {len(by_date)} dates")
print("dates with more than one asset:", {k: len(v) for k, v in by_date.items() if len(v) > 1})

In [ ]:
# The header says the two later versions list the same assets: compare paths and sizes with the pinned version
pinned = {a["path"]: a["size"] for a in assets}
for version in ("0.261015.2153", "0.261026.1912"):
    other = {a["path"]: a["size"] for a in d.list_assets(d.LINK_ID, version)}
    print(f"version {version}: {len(other)} assets; same paths as {d.LINK_VERSION}: {set(other) == set(pinned)}; "
          f"same sizes: {other == pinned}")

## Data check

Each development session is streamed once here for the checks; the fits below stream them again,
one process per session.

In [ ]:
sessions = [d.load_link_session(a["asset_id"], a["path"]) for a in dev_assets]
rows = []
for s in sessions:
    tr = s.trials
    dt = np.diff(s.timestamps)
    start_idx = np.array([int(np.argmin(np.abs(s.timestamps - t))) for t in tr.start_time])
    # usable-trial rule of link.py: the 25-bin window starting 8 bins after the start bin ends before start bin + trial_count
    usable = int(np.sum(start_idx + 8 + 24 < np.minimum(start_idx + tr.trial_count.to_numpy(), len(s.timestamps))))
    sbp, tcr = s.signals["SpikingBandPower"], s.signals["ThresholdCrossings"]
    rows.append(dict(
        date=s.date, task=s.task, target_style=",".join(sorted(tr.target_style.unique())),
        trials=len(tr), usable=usable, bins=len(s.timestamps),
        dt_ms_median=1000 * np.median(dt), dt_ms_min=1000 * dt.min(), dt_ms_max=1000 * dt.max(),
        trial_count_vs_duration=np.median(tr.trial_count / ((tr.stop_time - tr.start_time) / 0.02)),
        start_offset_ms=1000 * np.max(np.abs(s.timestamps[start_idx] - tr.start_time)),
        sbp_min=sbp.min(), sbp_max=sbp.max(), tcr_min=tcr.min(), tcr_max=tcr.max(),
        tcr_integer=bool(np.allclose(tcr, np.round(tcr))),
        pos_min=s.position.min(), pos_max=s.position.max(),
        vel_min=s.velocity.min(), vel_max=s.velocity.max(),
        targets=len(np.unique(np.c_[tr.index_target_position, tr.mrs_target_position], axis=0)),
        target_min=min(tr.index_target_position.min(), tr.mrs_target_position.min()),
        target_max=max(tr.index_target_position.max(), tr.mrs_target_position.max()),
    ))
check = pd.DataFrame(rows)
pd.set_option("display.width", 250, "display.max_columns", 40)
print(check.to_string(float_format=lambda v: f"{v:.4g}"))
print("units:", sessions[0].units)
print("tasks per paired date:", {dt: sorted(check.task[check.date == dt]) for dt in check.date.unique()})
print("target_style column agrees with the identifier's task:", bool((check.task == check.target_style).all()))

In [ ]:
# Recordings deposited more than once: compare every pair of sessions' SBP, TCR and position arrays
import hashlib
digests = {}
for s in sessions:
    for name, arr in [("SBP", s.signals["SpikingBandPower"]), ("TCR", s.signals["ThresholdCrossings"]), ("pos", s.position)]:
        digests.setdefault(name, []).append(hashlib.sha1(np.ascontiguousarray(arr).tobytes()).hexdigest())
for name, values in digests.items():
    dup = [(sessions[i].identifier, sessions[j].identifier) for i in range(len(values)) for j in range(i + 1, len(values)) if values[i] == values[j]]
    print(f"{name}: {len(set(values))} distinct arrays of {len(values)}; identical pairs: {dup}")
# near-duplicates: the largest correlation of the first 2000 SBP rows (channel mean) between sessions
first = np.array([s.signals["SpikingBandPower"][:2000].mean(axis=1) for s in sessions])
c = np.corrcoef(first)
np.fill_diagonal(c, np.nan)
print("largest between-session correlation of the first 2000 SBP bins:", round(float(np.nanmax(c)), 3))
# velocity against the first difference of position (velocity is described as derived post hoc)
for s in sessions[:18]:
    dp = np.diff(s.position, axis=0)
    r = [np.corrcoef(dp[:, k], s.velocity[1:, k])[0, 1] for k in range(2)]
    r_lag = [np.corrcoef(dp[:, k], s.velocity[:-1, k])[0, 1] for k in range(2)]
    ratio = [np.std(s.velocity[:, k]) / np.std(dp[:, k]) for k in range(2)]
    print(f"{s.identifier}: corr(velocity[t], position[t]-position[t-1]) = {r[0]:.3f}, {r[1]:.3f}; "
          f"with velocity[t-1] {r_lag[0]:.3f}, {r_lag[1]:.3f}; std ratio {ratio[0]:.2f}, {ratio[1]:.2f}")

In [ ]:
# Clock check: decode finger velocity in the 25-bin trial windows from 40 ms of SBP whose end is
# shifted by `lag` bins from the behaviour bin (positive lag = neural data earlier). If the SBP
# series and the trials table share a clock, held-out R2 peaks at a small positive lag and falls
# away on both sides; an offset between the streams would move the peak.
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
lags = [-25, -10, -5, -2, 0, 2, 5, 10, 25]
clock = []
for s in sessions:
    ds = d.build_link_dataset(s, context_mode="phase_only")  # the analysis windows: 25 bins from 8 bins after the start bin
    starts = np.array([int(np.argmin(np.abs(s.timestamps - t))) for t in s.trials.start_time.to_numpy()[ds.source_rows]])
    bins = starts[:, None] + 8 + np.arange(25)[None, :]
    sbp = s.signals["SpikingBandPower"]
    inside = (bins.min(axis=1) - 27 >= 0) & (bins.max(axis=1) + 25 < len(sbp))  # every lag stays inside the recording
    keep = np.flatnonzero(inside)
    position_of = {k: i for i, k in enumerate(keep)}
    tr = np.array([position_of[i] for i in ds.train_indices if inside[i]])
    te = np.array([position_of[i] for i in ds.test_indices if inside[i]])
    Y = ds.behavior[keep]
    row = {"session": s.identifier, "trials": int(inside.sum())}
    for lag in lags:
        X = np.stack([np.stack([sbp[b - lag - 2: b - lag].reshape(-1) for b in tb]) for tb in bins[keep]])
        m = make_pipeline(StandardScaler(), Ridge(alpha=1000.0)).fit(X[tr].reshape(-1, X.shape[-1]), Y[tr].reshape(-1, 2))
        row[lag] = r2_score(Y[te].reshape(-1, 2), m.predict(X[te].reshape(-1, X.shape[-1])), multioutput="variance_weighted")
    clock.append(row)
clock = pd.DataFrame(clock).set_index("session")
print("held-out R2 of velocity from 40 ms of SBP ending `lag` bins before the behaviour bin")
print(clock.round(3).to_string())
print("lag of the peak per session:", clock[lags].idxmax(axis=1).value_counts().to_dict())

In [ ]:
print(f"data check done in {time.time() - T0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## The fits

Every fit is the fully nested estimator (Methods 4.5; `du2026.fully_nested`, a port of
`evaluate_fully_nested_audit`) unless named otherwise, with the windows of the run scripts: 160 ms of
SBP or TCR ending at the behaviour bin, 100 ms of position history ending at the same bin, a 70/30
direction-stratified split with random_state 20260726. Per session:

- SBP, the five contexts of the ladder (`run_link_context_ladder.py`).
- SBP and TCR geometry by phase with 100 within-condition permutations, replacement ranks one to
  five and uniform random same-condition sources (Methods 4.7), and SBP geometry by phase and plus
  history with the three nonlinear-neural maps (Methods 4.6).
- TCR plus history.
- Neural-only Ridge (`decoders_nested`, the release's `evaluate_decoders_nested`, claims file).
- SBP geometry by phase under split seeds 20260727 and 20260728 (Results 2.7).
- Seed sensitivity (protocol, author step 3): SBP geometry by phase and plus history, and the
  neural-only R2, under split and refit seeds 20260726 plus 1000, 2000, 3000 and 4000.

In [ ]:
SEED = 20260726
EXTRA = (1000, 2000, 3000, 4000)
RANDOM_SOURCE_SEEDS = tuple(claim_seed("replacement-ranks-lower") + k for k in (0,) + EXTRA)
def job(name, signal, context, estimator="nested", seed=SEED, keep=False, **kw):
    build = dict(signal=signal, context_mode=context, seed=seed)
    est = dict(kw)
    if estimator == "nested":
        est["seed"] = seed
    if estimator == "decoders":
        est.update(seed=seed, neural_only_only=True)
    return (name, build, estimator, est, keep)

jobs = []
for context in d.LADDER:
    kw = {}
    if context == "geometry_tensor":
        kw = dict(n_permutations=100, permutation_seed_offsets=EXTRA, n_ranks=5,
                  random_source_seeds=RANDOM_SOURCE_SEEDS, rff_seeds=d.RFF_SEEDS)
    if context == "causal_history":
        kw = dict(rff_seeds=d.RFF_SEEDS)
    jobs.append(job(f"SBP_{context}", "SpikingBandPower", context, keep=(context == "geometry_tensor"), **kw))
jobs.append(job("TCR_geometry_tensor", "ThresholdCrossings", "geometry_tensor", n_permutations=100,
                permutation_seed_offsets=EXTRA, n_ranks=5, random_source_seeds=RANDOM_SOURCE_SEEDS))
jobs.append(job("TCR_causal_history", "ThresholdCrossings", "causal_history"))
for k in (0,) + EXTRA:
    jobs.append(job(f"neural_only_{SEED + k}", "SpikingBandPower", "geometry_tensor", "decoders", seed=SEED + k))
for seed in d.SPLIT_SEEDS[1:]:
    jobs.append(job(f"SBP_geometry_tensor_{seed}", "SpikingBandPower", "geometry_tensor", seed=seed))
for k in EXTRA:
    for context in ("geometry_tensor", "causal_history"):
        jobs.append(job(f"SBP_{context}_{SEED + k}", "SpikingBandPower", context, seed=SEED + k))

t = time.time()
results = d.run_parallel(d.link_session_jobs, [(a["asset_id"], a["path"], jobs) for a in dev_assets])
print(f"{len(jobs)} fits per session on {len(results)} sessions in {(time.time() - t) / 60:.1f} min")

def table(name, field="gain"):
    return pd.DataFrame([{"date": r["date"], "task": r["task"], field: r[name][field]} for r in results])
def task_means(name, field="gain"):
    return table(name, field).groupby("task")[field].mean()

## link-development-sample

> The LINK development set is the nine dates with both a CO and an RD session, 18 sessions and
> 6,750 usable trials from one macaque, with 375 usable trials in every analyzed session.

In [ ]:
usable = pd.DataFrame([{"date": r["date"], "task": r["task"], "usable": r["SBP_geometry_tensor"]["n_usable"],
                        "trials_table": r["SBP_geometry_tensor"]["n_trials_table"]} for r in results])
print(usable.to_string())
both = [dt for dt in paired_dates if sorted(check.task[check.date == dt]) == ["CO", "RD"]]
print(f"dates with both a CO and an RD session: {len(both)}  (expected 9)")
print(f"sessions: {len(usable)}; subjects in the asset paths: {sorted(set(a['path'].split('/')[0] for a in dev_assets))}")
print(f"usable trials: {usable.usable.sum()}  (expected 6750, tolerance 675)")
print(f"sessions with 375 usable trials: {(usable.usable == 375).sum()} of {len(usable)}; "
      f"range {usable.usable.min()} to {usable.usable.max()}; trials dropped by the window rule: "
      f"{(usable.trials_table - usable.usable).sum()}")

**Result: reproduced.** Nine dates have both a CO and an RD session (the only dates with more than one asset), and each of the 18 sessions has 375 trials in its table, all 375 usable under the window rule of link.py: 6,750 usable trials in total.

## link-neural-only-r2

> In the LINK development sessions, neural-only Ridge decoding of finger velocity from 160 ms of SBP
> had held-out R2 of 0.308 in CO and 0.313 in RD, the same at every rung of the context ladder.

X, Y and the split do not depend on the context (the split is stratified by direction octant,
which every rung shares), so one neural-only fit per session serves every rung.

In [ ]:
no = table(f"neural_only_{SEED}", "neural_r2")
print(no.pivot(index="date", columns="task", values="neural_r2").round(4).to_string())
m = no.groupby("task").neural_r2.mean()
print(f"CO mean {m['CO']:.4f} (expected 0.308 +- 0.0308); RD mean {m['RD']:.4f} (expected 0.313 +- 0.0313)")
for k in (0,) + EXTRA:
    mm = task_means(f"neural_only_{SEED + k}", "neural_r2")
    print(f"seed {SEED + k}: CO {mm['CO']:.4f}, RD {mm['RD']:.4f}")
print("neural-only penalties chosen:", collections.Counter(r[f"neural_only_{SEED}"]["neural_alpha"] for r in results))

**Result: reproduced.** Neural-only Ridge from 160 ms of SBP gives held-out R2 of 0.3077 in CO and 0.3129 in RD (paper 0.308 and 0.313). Under the four other split seeds the task means stay between 0.2969 and 0.3248, inside both tolerances.

## link-context-ladder-gain

> In LINK with SBP, mean context-conditional neural gain fell along the context ladder, from
> 0.3012/0.3125 (CO/RD) beyond phase alone to 0.0657/0.0502 beyond direction by phase, 0.0443/0.0332
> beyond additive geometry, 0.0193/0.0209 beyond geometry by phase, and 0.0077/0.0098 beyond
> geometry by phase plus measured-output history.

In [ ]:
expected = {"phase_only": (0.3012, 0.3125), "condition_phase": (0.0657, 0.0502), "geometry_additive": (0.0443, 0.0332),
            "geometry_tensor": (0.0193, 0.0209), "causal_history": (0.0077, 0.0098)}
ladder = []
for context in d.LADDER:
    m = task_means(f"SBP_{context}")
    g = table(f"SBP_{context}").gain
    for task, exp in zip(("CO", "RD"), expected[context]):
        ladder.append(dict(context=d.LADDER_NAMES[context], task=task, mean_gain=m[task], expected=exp,
                           within_10pct=abs(m[task] - exp) <= 0.1 * exp))
    print(f"{d.LADDER_NAMES[context]:>30}: session gains {g.min():.4f} to {g.max():.4f}, {int((g > 0).sum())}/18 positive")
ladder = pd.DataFrame(ladder)
print(ladder.to_string(float_format=lambda v: f"{v:.4f}"))
falls = {task: bool(np.all(np.diff(ladder[ladder.task == task].mean_gain.to_numpy()) < 0)) for task in ("CO", "RD")}
print("falls at every step:", falls, "; values within tolerance:", int(ladder.within_10pct.sum()), "of 10")
print("per session (rows: date and task; columns: rung):")
print(pd.DataFrame({d.LADDER_NAMES[c]: table(f"SBP_{c}").set_index(["date", "task"]).gain for c in d.LADDER}).round(4).to_string())

**Result: reproduced.** The task means are 0.3012/0.3125, 0.0657/0.0502, 0.0443/0.0332, 0.0193/0.0209 and 0.0077/0.0098 (CO/RD) along the five rungs, equal to the paper's values at the printed precision, so all 10 are within tolerance, and both task means fall at every step.

## link-context-only-r2

> Context-only R2 in LINK rose along the ladder from 0.0084/0.0055 (CO/RD) with phase alone to
> 0.6617/0.6839 with geometry by phase plus measured-output history.

In [ ]:
cr = pd.DataFrame({d.LADDER_NAMES[c]: task_means(f"SBP_{c}", "context_r2") for c in d.LADDER}).T
print(cr.round(4).to_string())
print(f"phase only: CO {cr.loc['phase only', 'CO']:.4f}, RD {cr.loc['phase only', 'RD']:.4f} (criterion below 0.02; 10% tolerances 0.0084 +- 0.00084, 0.0055 +- 0.00055)")
print(f"plus history: CO {cr.loc['plus measured-output history', 'CO']:.4f} (0.6617 +- 0.06617), RD {cr.loc['plus measured-output history', 'RD']:.4f} (0.6839 +- 0.06839)")
pc = table("SBP_phase_only", "context_r2").context_r2
print(f"phase-only session context R2 range {pc.min():.4f} to {pc.max():.4f}")

**Result: reproduced.** Context-only R2 is 0.0084/0.0055 (CO/RD) with phase only, below 0.02 and equal to the paper's values, and 0.6617/0.6839 with geometry by phase plus measured-output history, equal to the paper's.

## link-phase-only-replacement

> With phase-only context, the mean nearest-replacement gain in RD stayed positive (0.10498).

Under phase only every trial has the same context trajectory, so every distance is zero and the tie
rule (lower trial index, Methods 4.7) picks the first other same-direction test trial.

In [ ]:
pr = table("SBP_phase_only", "replacement_gain")
m = pr.groupby("task").replacement_gain.mean()
print(pr.pivot(index="date", columns="task", values="replacement_gain").round(4).to_string())
print(f"RD mean replacement gain, phase only: {m['RD']:.5f} (expected 0.10498 +- 0.010498); CO {m['CO']:.5f}")
print(f"RD sessions positive: {int((pr[pr.task == 'RD'].replacement_gain > 0).sum())}/9")

**Result: reproduced.** Under phase only, the mean nearest-replacement gain over the nine RD sessions is 0.10498 (paper 0.10498), positive in 8 of 9 RD sessions; in CO it is -0.01087.

## link-task-contrast-by-context

> The descriptive RD-minus-CO gain contrast changed with the context; it was positive under phase
> only, negative under direction by phase and additive geometry, near zero under geometry by phase,
> and small positive after measured-output history.

Per date RD minus CO, mean over nine dates, 95% interval from 10,000 date resamples (generator
seeded from the claim; the claim seed plus 1000 to 4000 give the interval's sensitivity).

In [ ]:
contrast = []
for context in d.LADDER:
    p = table(f"SBP_{context}").pivot(index="date", columns="task", values="gain")
    diff = (p["RD"] - p["CO"]).to_numpy()
    cis = []
    for k in (0,) + EXTRA:
        rng = np.random.default_rng(claim_seed("link-task-contrast-by-context") + k)
        boots = np.array([rng.choice(diff, len(diff), replace=True).mean() for _ in range(10000)])
        cis.append(np.quantile(boots, [0.025, 0.975]))
    contrast.append(dict(context=d.LADDER_NAMES[context], mean=diff.mean(), ci_low=cis[0][0], ci_high=cis[0][1],
                         ci_contains_zero=bool(cis[0][0] <= 0 <= cis[0][1]),
                         ci_low_range=f"{min(c[0] for c in cis):.4f}..{max(c[0] for c in cis):.4f}",
                         ci_high_range=f"{min(c[1] for c in cis):.4f}..{max(c[1] for c in cis):.4f}",
                         dates_positive=int((diff > 0).sum())))
contrast = pd.DataFrame(contrast).set_index("context")
print(contrast.to_string(float_format=lambda v: f"{v:.4f}"))
g = abs(contrast.loc["geometry by phase", "mean"])
print(f"reading (b): |geometry by phase| {g:.4f} < |direction by phase| {abs(contrast.loc['direction by phase', 'mean']):.4f} "
      f"and < |additive geometry| {abs(contrast.loc['additive geometry', 'mean']):.4f}: "
      f"{g < abs(contrast.loc['direction by phase', 'mean']) and g < abs(contrast.loc['additive geometry', 'mean'])}")

**Result: reproduced.** The mean RD-minus-CO contrast over nine dates is positive under phase only (0.0113), negative under direction by phase (-0.0155) and additive geometry (-0.0110), near zero under geometry by phase (0.0015; its date-bootstrap interval, -0.0035 to 0.0077, contains zero, and its size is below the two negative rungs, so both readings hold) and positive after measured-output history (0.0021). The phase-only and history intervals also contain zero; the claim is judged on the signs of the means, as the criterion says.

## nested-geometry-gain

> With geometry-by-phase context, fully nested mean gain over the 18 LINK sessions was 0.02009 for SBP
> (18 of 18 sessions positive) and 0.01155 for TCR (17 of 18 positive).

In [ ]:
def summary(name, field="gain", seed_key=None):
    t = table(name, field)
    by_date = {dt: g[field].to_numpy() for dt, g in t.groupby("date")}
    lo, hi = d.date_bootstrap(by_date, np.random.default_rng(claim_seed("nested-geometry-gain")))
    return dict(mean=t[field].mean(), positive=int((t[field] > 0).sum()), min=t[field].min(), max=t[field].max(),
                CO=t[t.task == "CO"][field].mean(), RD=t[t.task == "RD"][field].mean(), ci_low=lo, ci_high=hi)
for name, exp, exp_pos in [("SBP_geometry_tensor", 0.02009, 18), ("TCR_geometry_tensor", 0.01155, 17)]:
    s_ = summary(name)
    print(f"{name}: mean {s_['mean']:.5f} (expected {exp}), positive {s_['positive']}/18 (expected {exp_pos}), "
          f"range {s_['min']:.5f} to {s_['max']:.5f}, CO {s_['CO']:.5f}, RD {s_['RD']:.5f}, "
          f"date-clustered 95% interval {s_['ci_low']:.5f} to {s_['ci_high']:.5f}")
print("tolerances: SBP 0.00423, TCR 0.00446; counts 1.8 and 1.7")
vals = [np.mean(table("SBP_geometry_tensor" if k == 0 else f"SBP_geometry_tensor_{SEED + k}").gain) for k in (0,) + EXTRA]
pos = [int((table("SBP_geometry_tensor" if k == 0 else f"SBP_geometry_tensor_{SEED + k}").gain > 0).sum()) for k in (0,) + EXTRA]
print("SBP mean under seeds 20260726 +0/1000/2000/3000/4000:", np.round(vals, 5), "positive:", pos)
print(table("SBP_geometry_tensor").merge(table("TCR_geometry_tensor"), on=["date", "task"], suffixes=("_SBP", "_TCR")).round(5).to_string())

**Result: reproduced.** Under geometry by phase the mean gain is 0.02009 for SBP with 18 of 18 sessions positive, and 0.01155 for TCR with 17 of 18 positive, equal to the paper's values; the date-clustered intervals are 0.01586 to 0.02401 and 0.00711 to 0.01591. Under the four other split seeds the SBP mean is 0.01795 to 0.01918, all 18 sessions positive (TCR was not re-run under other seeds).

## nested-history-gain

> Adding lag-matched measured-output history reduced fully nested mean gain to 0.00873 for SBP (17 of
> 18 sessions positive) and 0.00420 for TCR (16 of 18 positive).

In [ ]:
for name, exp, exp_pos in [("SBP_causal_history", 0.00873, 17), ("TCR_causal_history", 0.0042, 16)]:
    s_ = summary(name)
    print(f"{name}: mean {s_['mean']:.5f} (expected {exp}), positive {s_['positive']}/18 (expected {exp_pos}), "
          f"range {s_['min']:.5f} to {s_['max']:.5f}, CO {s_['CO']:.5f}, RD {s_['RD']:.5f}, "
          f"date-clustered 95% interval {s_['ci_low']:.5f} to {s_['ci_high']:.5f}")
print("tolerances: SBP 0.00273, TCR 0.0021; counts 1.7 and 1.6")
for sig in ("SBP", "TCR"):
    print(f"{sig}: history mean {table(f'{sig}_causal_history').gain.mean():.5f} below geometry mean {table(f'{sig}_geometry_tensor').gain.mean():.5f}:",
          table(f"{sig}_causal_history").gain.mean() < table(f"{sig}_geometry_tensor").gain.mean())
vals = [np.mean(table("SBP_causal_history" if k == 0 else f"SBP_causal_history_{SEED + k}").gain) for k in (0,) + EXTRA]
pos = [int((table("SBP_causal_history" if k == 0 else f"SBP_causal_history_{SEED + k}").gain > 0).sum()) for k in (0,) + EXTRA]
print("SBP history mean under the five seeds:", np.round(vals, 5), "positive:", pos)

**Result: reproduced.** With measured-output history the mean gain is 0.00873 for SBP (17 of 18 positive) and 0.00420 for TCR (16 of 18 positive), equal to the paper's values and below each signal's geometry-by-phase mean. Under the four other seeds the SBP mean is 0.00742 to 0.00829, with 17 or 18 sessions positive.

## nested-replacement-negative

> Nearest source-context replacement made all four LINK mean gains negative, -0.00714 and -0.00407 for
> SBP (geometry by phase, plus history) and -0.00905 and -0.00537 for TCR.

In [ ]:
exp = {"SBP_geometry_tensor": -0.00714, "SBP_causal_history": -0.00407, "TCR_geometry_tensor": -0.00905, "TCR_causal_history": -0.00537}
within = 0
for name, e in exp.items():
    t = table(name, "replacement_gain").replacement_gain
    ok = abs(t.mean() - e) <= abs(0.1 * e)
    within += ok
    print(f"{name}: mean replacement gain {t.mean():.5f} (expected {e} +- {abs(0.1 * e):.6f}; within: {ok}); "
          f"negative sessions {int((t < 0).sum())}/18; range {t.min():.5f} to {t.max():.5f}")
print("all four negative:", all(table(n, "replacement_gain").replacement_gain.mean() < 0 for n in exp), "; within tolerance:", within, "of 4")
for k in EXTRA:
    print(f"seed {SEED + k}: SBP geometry {table(f'SBP_geometry_tensor_{SEED + k}', 'replacement_gain').replacement_gain.mean():.5f}, "
          f"SBP history {table(f'SBP_causal_history_{SEED + k}', 'replacement_gain').replacement_gain.mean():.5f}; within tolerance: "
          f"{[bool(abs(table(f'{n}_{SEED + k}', 'replacement_gain').replacement_gain.mean() - exp[n]) <= abs(0.1 * exp[n])) for n in ('SBP_geometry_tensor', 'SBP_causal_history')]}")

**Result: reproduced.** Nearest source-context replacement gives mean gains of -0.00714 and -0.00407 for SBP and -0.00905 and -0.00537 for TCR, all four negative and all within tolerance. The SBP values move with the split seed: under the four other seeds they are -0.00887 to -0.00714 (geometry) and -0.00496 to -0.00428 (history), so under seed plus 1000 both SBP values are outside their 10% tolerances, under plus 2000 the history value and under plus 3000 the geometry value is outside, and under plus 4000 both are inside.

## replacement-source-diagnostics

> For primary SBP geometry, every test trial had at least five eligible same-condition sources, the
> median over sessions of the maximum source reuse was three targets and the maximum over sessions
> four, and self-replacement reproduced the correct-pair predictions exactly.

In [ ]:
diag = pd.DataFrame([{"date": r["date"], "task": r["task"], "eligible_min": r["SBP_geometry_tensor"]["eligible_sources_min"],
                      "reuse_max": r["SBP_geometry_tensor"]["source_reuse_max"],
                      "self_error": r["SBP_geometry_tensor"]["self_replacement_max_abs_error"]} for r in results])
print(diag.to_string())
print(f"minimum eligible sources over sessions: {diag.eligible_min.min()} (expected at least 5)")
print(f"median over sessions of the maximum reuse: {diag.reuse_max.median()} (expected 3); maximum: {diag.reuse_max.max()} (expected 4)")
print(f"largest self-replacement difference: {diag.self_error.max():.3g} (criterion below 1e-10)")

**Result: reproduced.** The fewest eligible same-condition sources for any test trial is 5, the median over sessions of the largest reuse of one source is 3 and the largest over sessions is 4, and self-replacement reproduces the correct-pair predictions exactly (largest difference 0).

## permutation-negative

> The mean gain of 100 within-condition whole-trial permutations was negative in every LINK session,
> with an across-session mean of -0.0142.

In [ ]:
for label, names in [("(a) 18 SBP geometry sessions", ["SBP_geometry_tensor"]),
                     ("(b) 36 SBP and TCR geometry sessions", ["SBP_geometry_tensor", "TCR_geometry_tensor"])]:
    vals = np.concatenate([table(n, "permutation_mean").permutation_mean.to_numpy() for n in names])
    print(f"{label}: negative {int((vals < 0).sum())}/{len(vals)}, mean {vals.mean():.5f} (expected -0.0142 +- 0.00142), "
          f"range {vals.min():.5f} to {vals.max():.5f}")
    for k in EXTRA:
        v = np.concatenate([[r[n]["permutation_means"][k] for r in results] for n in names])
        print(f"   permutation generator offset +{k}: negative {int((v < 0).sum())}/{len(v)}, mean {v.mean():.5f}")

**Result: reproduced.** Under reading (a), the 18 SBP geometry-by-phase sessions, the permutation mean is negative in 18 of 18 sessions with an across-session mean of -0.01418 (paper -0.0142); reading (b), with the TCR sessions, also holds (36 of 36, -0.01387). Four other permutation generators give -0.01405 to -0.01426 under reading (a).

## replacement-penalty-distance

> The replacement penalty had no consistent rank relation to the nearest-source context distance across
> sessions.

In [ ]:
sp = table("SBP_geometry_tensor", "replacement_penalty_distance_spearman")
v = sp.replacement_penalty_distance_spearman
print(sp.round(3).to_string())
print(f"positive {int((v > 0).sum())}, negative {int((v < 0).sum())} of {len(v)}; mean {v.mean():.4f}")
by_date = {dt: g.replacement_penalty_distance_spearman.to_numpy() for dt, g in sp.groupby("date")}
for k in (0,) + EXTRA:
    lo, hi = d.date_bootstrap(by_date, np.random.default_rng(claim_seed("replacement-penalty-distance") + k))
    print(f"date-bootstrap 95% interval of the mean (generator +{k}): {lo:.4f} to {hi:.4f}; contains zero: {lo <= 0 <= hi}")
print("reading (a) not all of one sign:", bool((v > 0).any() and (v < 0).any()))

**Result: reproduced.** The per-session Spearman correlations between nearest-source distance and replacement penalty are positive in 8 and negative in 10 of 18 sessions (reading a), and their mean, 0.0058, has a date-bootstrap interval of -0.0204 to 0.0301 that contains zero (reading b), under all five generators.

## replacement-ranks-lower

> Replacement with the first to fifth nearest sources and with a random same-condition source all
> lowered the across-session mean gain.

The random source is a uniform draw among the other same-condition test trials (a standard choice,
claims file); five draws are shown, from the claim's seed and that seed plus 1000 to 4000.

In [ ]:
for sig in ("SBP", "TCR"):
    name = f"{sig}_geometry_tensor"
    correct = table(name).gain.mean()
    ranks = np.array([r[name]["rank_gains"] for r in results]).mean(axis=0)
    rand = np.array([r[name]["random_source_gains"] for r in results]).mean(axis=0)
    print(f"{sig} geometry: correct-pair mean {correct:.5f}; rank 1-5 means {np.round(ranks, 5)}; "
          f"random-source means over five draws {np.round(rand, 5)}")
    print(f"   all six (ranks and the first random draw) below the correct-pair mean: {bool(np.all(ranks < correct) and rand[0] < correct)}; "
          f"every random draw below: {bool(np.all(rand < correct))}")

**Result: reproduced.** For SBP the correct-pair mean is 0.02009 and replacement with ranks one to five gives -0.00714 to -0.01167, and random same-condition sources -0.01382 to -0.01433 over five draws, all lower (reading a); for TCR the same holds (0.01155 against -0.00905 to -0.01411).

## practical-mse-reduction

> In LINK, SBP geometry-by-phase gain reduced context-model MSE by 4.92% on average (mean context RMSE
> 0.01407 to 0.01372), adding measured-output history reduced the remaining MSE by 2.72%, and the TCR
> reductions were 2.83% and 1.31%.

In [ ]:
vals = [("SBP geometry % MSE reduction", table("SBP_geometry_tensor", "mse_reduction_percent").mse_reduction_percent.mean(), 4.92),
        ("SBP geometry mean context RMSE", table("SBP_geometry_tensor", "context_rmse").context_rmse.mean(), 0.01407),
        ("SBP geometry mean correct-pair RMSE", table("SBP_geometry_tensor", "exact_rmse").exact_rmse.mean(), 0.01372),
        ("SBP history % MSE reduction", table("SBP_causal_history", "mse_reduction_percent").mse_reduction_percent.mean(), 2.72),
        ("TCR geometry % MSE reduction", table("TCR_geometry_tensor", "mse_reduction_percent").mse_reduction_percent.mean(), 2.83),
        ("TCR history % MSE reduction", table("TCR_causal_history", "mse_reduction_percent").mse_reduction_percent.mean(), 1.31)]
within = 0
for label, v, e in vals:
    ok = abs(v - e) <= 0.1 * e
    within += ok
    print(f"{label}: {v:.5g} (expected {e} +- {0.1 * e:.4g}; within: {ok})")
print("within tolerance:", within, "of 6")
r = table("SBP_geometry_tensor", "mse_reduction_percent").mse_reduction_percent
print(f"SBP geometry session reductions {r.min():.2f} to {r.max():.2f}%")

**Result: reproduced.** The mean percentage MSE reductions are 4.92 (SBP geometry), 2.72 (SBP plus history), 2.83 (TCR geometry) and 1.31 (TCR plus history), and the SBP geometry mean RMSE falls from 0.01407 to 0.01372; all six values are within tolerance.

## representative-trial

> The prespecified rule for the illustrative trajectory (session with median primary SBP geometry
> gain, then the held-out trial with median context-minus-correct trial MSE improvement) selects
> 2022-09-21 RD, trial 123.

In [ ]:
geo = pd.DataFrame([{"date": r["date"], "task": r["task"], "gain": r["SBP_geometry_tensor"]["gain"], "i": i} for i, r in enumerate(results)])
geo = geo.sort_values(["gain", "date", "task"]).reset_index(drop=True)
print(geo.round(5).to_string())
lower = geo.iloc[(len(geo) - 1) // 2]
middle = geo.iloc[[(len(geo) - 1) // 2, len(geo) // 2]].sort_values(["date", "task"]).iloc[0]
for label, row in [("ninth in gain order (code: build_submission_figures.py figure8)", lower), ("earlier date of the two middle sessions", middle)]:
    r = results[int(row.i)]["SBP_geometry_tensor"]
    improvement = r["per_trial_context_mse"] - r["per_trial_exact_mse"]
    order = np.lexsort((r["test_indices"], improvement))
    position = int(order[(len(order) - 1) // 2])
    usable_index = int(r["test_indices"][position])
    # reading (c): the trial's position among the held-out trials in ascending trial order (test_indices
    # itself is in train_test_split's shuffled order, so an index into it is not a position)
    held_out_position = int(np.searchsorted(np.sort(r["test_indices"]), usable_index))
    print(f"{label}: {row.date} {row.task}; median trial: usable-trial index {usable_index}, "
          f"trials-table row {int(r['source_rows'][usable_index])}, position among the {len(order)} held-out trials "
          f"in trial order {held_out_position} (from 0)")

**Result: reproduced.** Taking the ninth of the 18 sessions in gain order (the lower middle session, as the release's figure code does) selects 2022-09-21 RD, and its median-improvement held-out trial is trial 123 both as the row of the NWB trials table (reading a) and as the usable-trial index (reading b), which coincide because no trial is dropped. Under reading (c), its position among the 113 held-out trials in trial order is 41 counting from 0 (the 42nd), not 123. The other reading of the session tie (the earlier date of the two middle sessions) selects 2021-10-15 RD, trial 260, so the match rests on the first reading of the tie rule.

## split-seed-stability

> The fully nested SBP geometry-by-phase result was stable across three whole-trial split and
> refitting seeds, with CO means 0.0193, 0.0180 and 0.0158 and RD means 0.0209, 0.0223 and 0.0201.

In [ ]:
exp = {20260726: (0.0193, 0.0209), 20260727: (0.0180, 0.0223), 20260728: (0.0158, 0.0201)}
within, positive = 0, True
for seed in d.SPLIT_SEEDS:
    name = "SBP_geometry_tensor" if seed == SEED else f"SBP_geometry_tensor_{seed}"
    m = task_means(name)
    for task, e in zip(("CO", "RD"), exp[seed]):
        ok = abs(m[task] - e) <= 0.1 * e
        within += ok
        positive &= m[task] > 0
        print(f"seed {seed} {task}: {m[task]:.4f} (expected {e} +- {0.1 * e:.5f}; within: {ok})")
print("all six positive:", bool(positive), "; within tolerance:", within, "of 6")

**Result: reproduced.** Under split seeds 20260726, 20260727 and 20260728 the CO means are 0.0193, 0.0180 and 0.0158 and the RD means 0.0209, 0.0223 and 0.0201, equal to the paper's six values and all positive.

## rff-link

> In LINK SBP, nonlinear-neural geometry-by-phase means were positive for both tasks under every map
> (combined range 0.00310 to 0.00404), and the measured-output-history mean ranged from 0.00111 to
> 0.00142.

In [ ]:
rows_ = []
for seed in d.RFF_SEEDS:
    for context in ("geometry_tensor", "causal_history"):
        t = pd.DataFrame([{"task": r["task"], "gain": r[f"SBP_{context}"]["rff"][seed]["gain"]} for r in results])
        rows_.append(dict(map_seed=seed, context=d.LADDER_NAMES[context], CO=t[t.task == "CO"].gain.mean(),
                          RD=t[t.task == "RD"].gain.mean(), all18=t.gain.mean(), positive=int((t.gain > 0).sum())))
rff = pd.DataFrame(rows_)
print(rff.to_string(float_format=lambda v: f"{v:.5f}"))
geo_ = rff[rff.context == "geometry by phase"]
his_ = rff[rff.context == "plus measured-output history"]
lo_g, hi_g = 0.0031 - 0.00031, 0.00404 + 0.000404
lo_h, hi_h = 0.00111 - 0.000111, 0.00142 + 0.000142
print(f"geometry CO and RD positive under every map: {bool((geo_[['CO', 'RD']] > 0).all().all())}")
print(f"reading (a), 18-session geometry means {geo_.all18.min():.5f} to {geo_.all18.max():.5f}, inside {lo_g:.5f}..{hi_g:.5f}: {bool(geo_.all18.between(lo_g, hi_g).all())}")
six = np.r_[geo_.CO, geo_.RD]
print(f"reading (b), six task means {six.min():.5f} to {six.max():.5f}, inside: {bool(np.all((six >= lo_g) & (six <= hi_g)))}")
print(f"history 18-session means {his_.all18.min():.5f} to {his_.all18.max():.5f}, inside {lo_h:.6f}..{hi_h:.6f}: {bool(his_.all18.between(lo_h, hi_h).all())}")
print("RFF penalties chosen (geometry):", collections.Counter(r["SBP_geometry_tensor"]["rff"][s]["alpha"] for r in results for s in d.RFF_SEEDS))
print(f"notebook run time {(time.time() - T0) / 60:.1f} min; peak memory of this process {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

**Result: reproduced.** With the three nonlinear-neural maps, the CO and RD geometry-by-phase means are positive under every map (0.00269 to 0.00417). Under reading (a) the 18-session geometry means span 0.00310 to 0.00404 and the history means 0.00111 to 0.00142, equal to the paper's ranges. Reading (b), the six task means, does not hold: the lowest, 0.00269, is below the widened bound of 0.00279.

## Summary

| Claim | Outcome |
|---|---|
| link-development-sample | reproduced |
| link-neural-only-r2 | reproduced |
| link-context-ladder-gain | reproduced |
| link-context-only-r2 | reproduced |
| link-phase-only-replacement | reproduced |
| link-task-contrast-by-context | reproduced |
| nested-geometry-gain | reproduced |
| nested-history-gain | reproduced |
| nested-replacement-negative | reproduced (seed sensitive) |
| replacement-source-diagnostics | reproduced |
| permutation-negative | reproduced |
| replacement-penalty-distance | reproduced |
| replacement-ranks-lower | reproduced |
| practical-mse-reduction | reproduced |
| representative-trial | reproduced (on one reading of the session tie rule) |
| split-seed-stability | reproduced |
| rff-link | reproduced (reading a) |

How the analysis differs from the paper's: the estimators are ports of the paper's released code
(commit b042fbe) and read the NWB files by streaming them; the bootstrap intervals use generators
seeded from the claims, not the release's seed 20260802; the seed sensitivity of the protocol was run
for the SBP geometry-by-phase and plus-history fits and the neural-only R2, not for the other ladder
rungs or for TCR.

What was found about the dataset: the 18 development sessions are as the paper describes (nine dates,
375 usable trials each, one subject); no session is deposited twice; SBP, TCR, position and velocity
share one 20 ms clock (velocity decoding from SBP peaks at lag 0 in 16 sessions and 2 bins in 2);
the released velocity is the forward first difference of the released position
(velocity[t] = position[t+1] - position[t]: correlation 1.000 and SD ratio 1.00 for both components in
17 sessions, and 1.000 and 0.987 in 2021-10-15 CO, the session with the timestamp gap), which the files
do not state; the
threshold-crossing series is labelled in volts though it holds integer counts (0 to 11 per bin).
One session (2021-10-15 CO) has a gap of 1094 ms in its 20 ms timestamps.